# Boat Routing  

This notebook contains the heuristic routing of the boat to the top *n* predicted most polluted beaches. It follows the steps below:

1. Create a massive distance matrix of all the beaches.
2. Filter out the top *n* most polluted beaches.

   **NOTE:** The current notebook uses the actual Total Weight from the FeatureSet. Next step includes integrating the prediction.
5. Extract Monte Carlo blueprints, using score metrics.
6. For the least distance 5% of routes, perform local search to minimize the distance.
7. Select the most efficient route and return it. 

## 1. Setup 

In [89]:
import pandas as pd
import numpy as np
import random
import TwoStepPipeline
import searoute as sr
from joblib import Parallel, delayed
import pyarrow as pa
import pyarrow.parquet as pq
import os


#### Data

There are two sources of data we must integrate: the hubs and the generated data from Task 1. We will integrate a pipeline that will result in the generated data. 

Operating under the following hub assumptions:
- There is one main hub at the Lavrio Marina
- Coordinates for the Marina are 37.7188° N, 24.05663° E in decimal degrees (src: [https://www.predictwind.com/marinas/greece/attica/lavrion-marina])
- When the ship visits the port, the trash is removed and the ship refueled
- The ship refuels once every 3 months, approximately
- The ship visits sections of the map rather than everything at once

Generated data must include the following features:
- Reward - estimated waste or other assigned reward for visiting the island
- Latitude - coordinate of the beach (DD)
- Longitude - coordinate of the beach (DD)
- Special Number - beach identification number

In [90]:
# hub data: update if the coordinates change
hub_coords = pd.DataFrame(data={"Special Number": "PORT", "Beach":'Λαύριο', "Total Weight":0, "Latitude":[37.7188], "Longitude":[24.05663], "Coords":[(24.05663, 37.7188)]})
print(hub_coords.head())

# [this is where we insert the pipeline for generation]
"""
PIPELINE HERE

** NOTE: we are using the Task1/0. Cleaning/FeatureSets/FeatureSet1.csv for code validation. 

"""

# assuming we have the generated data:
"""
This is where we insert the generated data as "beach".
* please note: the generated data needs to take the form of a dataframe with the following features:
    - Estimated Waste - waste expected to be present at any given beach 
    - Latitude - coordinates (DD)
    - Longitude - coordinates (DD)
    - Special Number - beach identification number
"""

beach = pd.read_csv("tempdata_fs1.csv")
beach = beach[['Special Number', 'Total Weight', 'Latitude', 'Longitude']]

other = pd.read_csv('beach_name_spn.csv')
spn_name_map = other[['Special Number', 'Beaches']]

beach['Beach'] = spn_name_map['Beaches']

beach = beach[:100]

  Special Number   Beach  Total Weight  Latitude  Longitude  \
0           PORT  Λαύριο             0   37.7188   24.05663   

                Coords  
0  (24.05663, 37.7188)  


In [67]:
# !pip install searoute
import searoute as sr

In [91]:
## LOCATION EXTRACTION
"""
For each beach, extract the coordinates as (lon, lat) pairs. 
This is the format searoute uses.
"""
beach['Coords'] = list(zip(beach.Longitude, beach.Latitude))

# ADD THE PORT

beach = pd.concat([hub_coords, beach], ignore_index=True)
beach.head()

,Special Number,Beach,Total Weight,Latitude,Longitude,Coords
0,PORT,Λαύριο,0,37.718800,24.056630,"(24.05663, 37.7188)"
1,1,Πάτροκλος Παραλία 1,323,37.656139,23.956917,"(23.956917, 37.656139)"
2,2,Πάτροκλος Παραλία 2 (1),945,37.656500,23.950917,"(23.950917, 37.6565)"
3,3,Πάτροκλος Παραλία 3 (1),420,37.655944,23.946500,"(23.9465, 37.655944)"
4,4,Πάτροκλος Παραλία 4,239,37.656194,23.944250,"(23.94425, 37.656194)"


### PARQUET DISTANCE MATRIX

In [81]:
# import numpy as np
# import pandas as pd
# import searoute as sr
# from joblib import Parallel, delayed
# import pyarrow as pa
# import pyarrow.parquet as pq
# import os

# # ========================================================
# # 1. SETUP DATA AND COMPUTE INSTANT VECTORIZED HAVERSINE
# # ========================================================
# names = beach['Special Number'].tolist()
# n = len(names)
# coord_array = beach[['Longitude', 'Latitude']].to_numpy()

# print(f"Initializing data for {n:,} sites...")
# print(f"Computing {(n**2)/2} baseline Haversine paths via matrix math...")

# lon = np.radians(coord_array[:, 0])
# lat = np.radians(coord_array[:, 1])

# delta_lon = lon[:, np.newaxis] - lon
# delta_lat = lat[:, np.newaxis] - lat

# a = np.sin(delta_lat / 2.0)**2 + np.cos(lat[:, np.newaxis]) * np.cos(lat) * np.sin(delta_lon / 2)**2
# c_rad = 2 * np.arcsin(np.sqrt(a))

# EARTH_RADIUS_KM = 6371.3
# haversine_matrix = EARTH_RADIUS_KM * c_rad

# # ========================================================
# # 2. CATEGORIZE COHORTS BY DISTANCE THRESHOLD
# # ========================================================
# CUTOFF_KM = 150.0  # Sweet spot for localized marine detours

# searoute_tasks = []
# static_haversine_pairs = []

# print(f"Filtering pairs using a {CUTOFF_KM} km threshold...")
# for i in range(n):
#     name_i = names[i]
#     lon_i, lat_i = coord_array[i, 0], coord_array[i, 1]
    
#     for j in range(i + 1, n):
#         straight_dist = haversine_matrix[i, j]
#         name_j = names[j]
        
#         if straight_dist <= CUTOFF_KM:
#             # High priority: complex coastal geometries/islands
#             searoute_tasks.append((name_i, name_j, (lon_i, lat_i), (coord_array[j, 0], coord_array[j, 1])))
#         else:
#             # Long-haul ocean transit: Haversine is a highly reliable proxy
#             static_haversine_pairs.append({'Origin': name_i, 'Destination': name_j, 'Distance_KM': straight_dist})

# print(f"→ Allocated {len(static_haversine_pairs):,} pairs directly to Haversine cache.")
# print(f"→ Allocated {len(searoute_tasks):,} pairs to true maritime parallel routing.")

# # ========================================================
# # 3. STREAM HAVERSINE BASELINE STRATUM TO DISK
# # ========================================================
# OUTPUT_FILE = 'master_hybrid_maritime_matrix.parquet'

# print(f"\nWriting baseline Haversine tracks directly to '{OUTPUT_FILE}'...")
# df_hav = pd.DataFrame(static_haversine_pairs)
# df_hav['Origin'] = df_hav['Origin'].astype(str)
# df_hav['Destination'] = df_hav['Destination'].astype(str)
# table_hav = pa.Table.from_pandas(df_hav)
# pq.write_table(table_hav, OUTPUT_FILE)

# # Memory cleanup optimization
# del static_haversine_pairs, df_hav

# # ========================================================
# # 4. RUN PARALLEL SEAROUTE FOR LOCAL SHORELINE INTERSECTIONS
# # ========================================================
# def compute_searoute_pair(name_i, name_j, origin, destination):
#     try:
#         route = sr.searoute(origin, destination, units='km')
#         return {'Origin': name_i, 'Destination': name_j, 'Distance_KM': route['properties']['length']}
#     except Exception:
#         # Fallback to Haversine if searoute hits a landlocked node error
#         return {'Origin': name_i, 'Destination': name_j, 'Distance_KM': np.inf}

# BATCH_SIZE = 30000
# writer = None

# print(f"\nProcessing remaining maritime loops via multi-core engine...")

# for b_start in range(0, len(searoute_tasks), BATCH_SIZE):
#     b_end = min(b_start + BATCH_SIZE, len(searoute_tasks))
#     batch_tasks = searoute_tasks[b_start:b_end]
    
#     # Run batch across all available CPU cores
#     batch_results = Parallel(n_jobs=-1, backend="threading")(
#         delayed(compute_searoute_pair)(p[0], p[1], p[2], p[3]) for p in batch_tasks
#     )
    
#     batch_df = pd.DataFrame(batch_results)

#     batch_df['Origin'] = batch_df['Origin'].astype(str)
#     batch_df['Destination'] = batch_df['Destination'].astype(str)
    
#     batch_table = pa.Table.from_pandas(batch_df)
    
#     if writer is None:
#         # Initialize the append-stream schema using PyArrow
#         writer = pq.ParquetWriter(OUTPUT_FILE, batch_table.schema)
        
#     writer.write_table(batch_table)
    
#     pct = (b_end / len(searoute_tasks)) * 100
#     print(f" Progress: {b_end:,}/{len(searoute_tasks):,} ({pct:.2f}%) saved securely to cache.")

# if writer:
#     writer.close()

# print(f"\nSuccess! Unified master hybrid matrix generated at '{OUTPUT_FILE}'.")

Initializing data for 9,300 sites...
Computing 43245000.0 baseline Haversine paths via matrix math...
Filtering pairs using a 150.0 km threshold...
→ Allocated 34,745,160 pairs directly to Haversine cache.
→ Allocated 8,495,190 pairs to true maritime parallel routing.

Writing baseline Haversine tracks directly to 'master_hybrid_maritime_matrix.parquet'...

Processing remaining maritime loops via multi-core engine...
 Progress: 30,000/8,495,190 (0.35%) saved securely to cache.
 Progress: 60,000/8,495,190 (0.71%) saved securely to cache.
 Progress: 90,000/8,495,190 (1.06%) saved securely to cache.
 Progress: 120,000/8,495,190 (1.41%) saved securely to cache.
 Progress: 150,000/8,495,190 (1.77%) saved securely to cache.
 Progress: 180,000/8,495,190 (2.12%) saved securely to cache.
 Progress: 210,000/8,495,190 (2.47%) saved securely to cache.
 Progress: 240,000/8,495,190 (2.83%) saved securely to cache.
 Progress: 270,000/8,495,190 (3.18%) saved securely to cache.
 Progress: 300,000/8,495

In [92]:
## COST EXTRACTION 
"""
Calculate the cost matrix (distance). 

Please note: 
    - To reduce the computational time and power required to run this, we first
        cluster the top 25% of beaches evaluated spatially.
***POSSIBLY: Incorporate a buffer: minimum distance of X km from the shoreline.
    - use this buffer to reproject the coordinates with the knowledge that the 
        tenders leave from the ship to close the gap
    - this buffer is necessary to ensure that the model relies on searoutes
         the ship is able to traverse, not interfering with land
"""


def calculate_dist(df_subset):
    """
    Calculates the distance matrix via SeaRoute library for the given df subset.

    df_subset: a cluster subset of the larger dataframe consisting of columns: 
    ['Special Number', 'Latitude', 'Longitude', 'Beach']
    
    """
    names = df_subset['Special Number'].tolist()
    n = len(names)
    coord_array = df_subset[['Longitude', 'Latitude']].to_numpy()

    c = pd.DataFrame(np.inf, index=names, columns=names)
    np.fill_diagonal(c.values, 0.0)
    
    for i in range(n):
        for j in range(i + 1, n):
            try:
                origin = (coord_array[i, 0], coord_array[i, 1])
                destination = (coord_array[j, 0], coord_array[j, 1])
                        
                route = sr.searoute(origin, destination, units='km')
                true_sea_dist = route['properties']['length']
        
                node_i = names[i]
                node_j = names[j]
        
                c.loc[node_i, node_j] = true_sea_dist
                c.loc[node_j, node_i] = true_sea_dist
    
            except Exception:
                pass


print("Final Maritime Cost Matrix (c_ij):")
print(c)


Final Maritime Cost Matrix (c_ij):
                 1           2           3           4           5          6  \
Origin                                                                          
1         0.000000    0.000000    0.000000    0.000000   10.472622  55.420661   
2         0.000000    0.000000    0.000000    0.000000   10.472622  55.420661   
3         0.000000    0.000000    0.000000    0.000000   10.472622  55.420661   
4         0.000000    0.000000    0.000000    0.000000   10.472622  55.420661   
5        10.472622   10.472622   10.472622   10.472622    0.000000  65.363449   
...            ...         ...         ...         ...         ...        ...   
96             inf         inf         inf         inf         inf        inf   
97             inf         inf         inf         inf  151.486635        inf   
98             inf         inf         inf         inf  151.486635        inf   
99      123.697334  123.697334  123.697334  123.697334  113.224712        

In [94]:
def get_routing_matrix(active_ids, parquet_path='master_hybrid_maritime_matrix.parquet'):
    """
    Extracts the distance matrix for the given beaches.
    """
    active_ids = [str(x) for x in active_ids]

    print(f"Loading master matrix from disk...")

    master_edges = pd.read_parquet(parquet_path)
    master_edges['Origin'] = master_edges['Origin'].astype(str)
    master_edges['Destination'] = master_edges['Destination'].astype(str)

    print(f"Slicing network for {len(active_ids):,} active nodes...")

    filtered_edges = master_edges[
        master_edges['Origin'].isin(active_ids) &
        master_edges['Destination'].isin(active_ids)
    ]

    # Pivot from 3 columns to a matrix
    c_subset = filtered_edges.pivot(index='Origin', columns='Destination', values='Distance_KM')
    c_subset = c_subset.reindex(index=active_ids, columns=active_ids)
    c_subset = c_subset.combine_first(c_subset.T)
    np.fill_diagonal(c_subset.values, 0.0)
    c_subset = c_subset.fillna(np.inf)
    print(f"Done!")

    return c_subset

c = get_routing_matrix(beach["Special Number"])


Loading master matrix from disk...
Slicing network for 101 active nodes...
Done!


### Monte Carlo Blueprints

1. select top *k* closest destinations (k = 3 arbitrary)
2. randomly select one of them 
3. "move" there
4. repeat

In [95]:
def generate_tsp_blueprint(all_nodes, matrix, k=3):

    # initialize
    unvisited = [node for node in all_nodes if node != "PORT"]
    current_node = "PORT"
    route = ["PORT"]
    total_distance = 0.0

    while unvisited:
        distance_row = matrix.loc[current_node]
        valid_options = []

        for j in unvisited:
            dist = distance_row[j]
            if np.isinf(dist):
                continue

            score = 1.0 / dist if dist > 0 else float('inf')
            valid_options.append({"beach":j, "score":score, "distance":dist})

        if not valid_options:
            break
        valid_options.sort(key=lambda x: x["score"], reverse=True)
        current_k = min(k, len(valid_options))
        selected = random.choice(valid_options[:current_k])

        next_beach = selected["beach"]
        unvisited.remove(next_beach)
        total_distance += selected["distance"]
        route.append(next_beach)
        current_node = next_beach

    total_distance += matrix.loc[current_node, "PORT"]
    route.append("PORT")

    return route, total_distance

In [87]:
# Local search helper functions
def calculate_route_dist(route, matrix):
    """total distance for route sequence"""
    total_dist = 0.0
    for i in range(len(route) - 1):
        total_dist += matrix.loc[route[i], route[i+1]]
    return total_dist

def two_opt_swap(route, i, j):
    """slice route, reverse inner segement"""
    new_route = route[:i] + route[i:j+1][::-1] + route[j+1:]
    return new_route

In [101]:
# LOCAL SEARCH

def run_2opt(route, matrix):
    best_route = route
    best_distance = calculate_route_dist(route, matrix)
    improvement = True

    while improvement:
        improvement = False

        for i in range(1, len(best_route) - 2):
            for j in range(i+1, len(best_route) -1):
                new_route = two_opt_swap(best_route, i, j)
                new_dist = calculate_route_dist(new_route, matrix)

                if new_dist < best_distance:
                    best_route = new_route
                    best_distance = new_dist
                    improvement = True

                    break
            if improvement:
                break

    return best_route, best_distance


# BUFFER WRAPPER

def run_tsp_adaptive_filter(matrix, iterations=1000, buffer_pct=.1, k=3):
    all_nodes = matrix.index.tolist()

    print(f"--- Stage 1: Generating {iterations} Raw Blueprints ---")
    raw_blueprints = []

    for i in range(iterations):
        raw_route, raw_dist = generate_tsp_blueprint(all_nodes, matrix, k=k)
        raw_blueprints.append({"route":raw_route, "distance":raw_dist})

    raw_blueprints.sort(key=lambda x: x["distance"])

    best_raw_dist = raw_blueprints[0]["distance"]
    max_acceptable_dist = best_raw_dist * (1.0 + buffer_pct)

    elite_blueprints = [b for b in raw_blueprints if b["distance"] <= max_acceptable_dist]
    print(f"\n--- FILTER RESULT ---")
    print(f"Absolute Best Raw Baseline: {best_raw_dist:.2f} km")
    print(f"Quality Buffer Threshold ({buffer_pct*100}%): {max_acceptable_dist:.2f} km")
    print(f"Slashed candidate pool from {iterations} down to {len(elite_blueprints)} elite routes.")
    
    print(f"\n--- STAGE 2: 2-Opt on Top Pool ---")
    best_route = None
    best_distance = float('inf')
    
    for idx, candidate in enumerate(elite_blueprints):
        opt_route, opt_dist = run_2opt(candidate["route"], matrix)
        
        if opt_dist < best_distance:
            best_distance = opt_dist
            best_route = opt_route
            best_orig = candidate["route"]
            orig_dist = calculate_route_dist(candidate["route"], matrix)
            print(f"  → Success! Candidate #{idx+1} untwisted to create a new global champion: {best_distance:.2f} km")
            
    return best_route, best_distance, best_orig, orig_dist

best_blueprint, shortest_dist, original_route, original_distance = run_tsp_adaptive_filter(c)

print("\n" + "="*50)
print(f"OPTIMIZATION COMPLETE")
print(f"Original Route Found: {original_distance:.2f} km")
print("Original Sequence: " + " -> ".join([str(node) for node in original_route]))

print(f"Absolute Best Integrated Route Found: {shortest_dist:.2f} km")
print("Optimized Sequence: " + " -> ".join([str(node) for node in best_blueprint]))
print("="*50)

        

--- Stage 1: Generating 1000 Raw Blueprints ---

--- FILTER RESULT ---
Absolute Best Raw Baseline: 590.71 km
Quality Buffer Threshold (50.0%): 886.06 km
Slashed candidate pool from 1000 down to 16 elite routes.

--- STAGE 2: 2-Opt on Top Pool ---
  → Success! Candidate #1 untwisted to create a new global champion: 444.65 km

OPTIMIZATION COMPLETE
Original Route Found: 590.71 km
Original Sequence: PORT -> 1 -> 3 -> 2 -> 5 -> 34 -> 33 -> 37 -> 36 -> 29 -> 30 -> 31 -> 39 -> 40 -> 41 -> 38 -> 43 -> 45 -> 42 -> 46 -> 47 -> 32 -> 48 -> 44 -> 35 -> 4 -> 15 -> 16 -> 21 -> 14 -> 23 -> 22 -> 25 -> 24 -> 26 -> 28 -> 18 -> 7 -> 9 -> 6 -> 11 -> 10 -> 12 -> 8 -> 17 -> 19 -> 13 -> 20 -> 27 -> PORT
Absolute Best Integrated Route Found: 444.65 km
Optimized Sequence: PORT -> 2 -> 3 -> 1 -> 4 -> 15 -> 16 -> 21 -> 14 -> 23 -> 22 -> 18 -> 28 -> 26 -> 24 -> 25 -> 27 -> 20 -> 13 -> 19 -> 17 -> 8 -> 12 -> 10 -> 11 -> 6 -> 9 -> 7 -> 5 -> 34 -> 33 -> 37 -> 36 -> 35 -> 32 -> 48 -> 44 -> 47 -> 46 -> 42 -> 45 -> 4

# Mapping

In [102]:
import folium
import pandas as pd
import json

# ==========================================
# 1. GENERATE ORDERED COORDINATE DATA
# ==========================================
# Replace 'beach' with your actual beach DataFrame name
coordinate_lookup = beach.set_index('Special Number')[['Latitude', 'Longitude', 'Beach']].to_dict('index')

port_lat = 37.7188
port_lon = 24.05663 
# coordinate_lookup['PORT'] = {'Latitude': port_lat, 'Longitude': port_lon}

# Build an ordered array of dicts for our JavaScript engine to parse
path_data = []
for idx, node in enumerate(best_blueprint):
    path_data.append({
        "step": idx,
        "num": str(node),
        "name": coordinate_lookup[node]['Beach'],
        "lat": coordinate_lookup[node]['Latitude'],
        "lon": coordinate_lookup[node]['Longitude']
    })

# Convert to JSON string for seamless injection into JavaScript
json_path_data = json.dumps(path_data)

# ==========================================
# 2. INITIALIZE THE BASICS
# ==========================================
m = folium.Map(location=[port_lat, port_lon], zoom_start=9, control_scale=True)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(m)

# Plot the starting Home Port immediately as our anchor point
folium.Marker(
    location=[port_lat, port_lon],
    popup="<b>STARTING HOME PORT</b>",
    icon=folium.Icon(color="gold", icon="home", prefix="fa")
).add_to(m)

# ==========================================
# 3. CUSTOM INJECTED JAVASCRIPT DASHBOARD
# ==========================================
# We construct a floating control panel widget and custom logic
clickthrough_script = f"""
<div id="control-panel" style="
    position: fixed; 
    bottom: 30px; 
    left: 30px; 
    width: 280px;
    background-color: white; 
    padding: 15px; 
    border-radius: 8px; 
    box-shadow: 0 0 15px rgba(0,0,0,0.2); 
    z-index: 9999; 
    font-family: Arial, sans-serif;
">
    <h4 style="margin-top: 0; margin-bottom: 10px; color: #333;">TSP Route Sequencer</h4>
    <p style="margin: 5px 0; font-size: 13px;"><b>Current Step:</b> <span id="step-display">0 / {len(path_data)-1}</span></p>
    <p style="margin: 5px 0; font-size: 13px;"><b>Location:</b> <span id="loc-display">PORT</span></p>
    <p style="margin: 5px 0; font-size: 13px;"><b>Number:</b> <span id="spn-display">0</span></p>
    <hr style="border: 0; border-top: 1px solid #ccc; margin: 10px 0;">
    <button id="next-btn" style="
        width: 100%; 
        background-color: #007bff; 
        color: white; 
        border: none; 
        padding: 8px; 
        border-radius: 4px; 
        cursor: pointer; 
        font-weight: bold;
    ">Next Leg →</button>
</div>

<script>
document.addEventListener("DOMContentLoaded", function() {{
    // 1. Load the backend routing sequence data
    const fullPath = {json_path_data};
    let currentStepIndex = 0;
    
    // Access the Leaflet map global object instanced by folium
    const mapObject = {m.get_name()};
    
    // Keep trackers for active layer objects so they render cleanly
    let activeLines = [];
    let activeMarkers = [];

    document.getElementById("next-btn").addEventListener("click", function() {{
        if (currentStepIndex >= fullPath.length - 1) {{
            alert("Sailing itinerary complete! The ship has successfully returned to port.");
            return;
        }}
        
        // Move tracking pointer forward
        currentStepIndex++;
        
        const originNode = fullPath[currentStepIndex - 1];
        const targetNode = fullPath[currentStepIndex];
        
        // Update DOM metrics on the dashboard panel
        document.getElementById("step-display").innerText = currentStepIndex + " / " + (fullPath.length - 1);
        document.getElementById("loc-display").innerText = targetNode.name;
        document.getElementById("spn-display").innerText = targetNode.num;
        
        // 2. Plot the new connecting leg
        const lineCoords = [
            [originNode.lat, originNode.lon],
            [targetNode.lat, targetNode.lon]
        ];
        
        let polyline = L.polyline(lineCoords, {{
            color: '#007bff',
            weight: 4,
            opacity: 0.85,
            dashArray: '5, 10' // Renders as dashed lines to visually imply transit pathing
        }}).addTo(mapObject);
        activeLines.push(polyline);
        
        // 3. Mark the destination stop node
        let markerColor = (targetNode.name === 'PORT') ? 'lightblue' : 'blue';
        let circle = L.circleMarker([targetNode.lat, targetNode.lon], {{
            radius: 7,
            color: 'black',
            weight: 1,
            fillColor: markerColor,
            fillOpacity: 0.9
        }}).addTo(mapObject);
        
        circle.bindPopup("<b>Stop #" + currentStepIndex + "</b><br>ID: " + targetNode.name);
        activeMarkers.push(circle);
        
        // Smoothly pan the viewport camera to focus on the newly added leg segment
        mapObject.panTo([targetNode.lat, targetNode.lon]);
    }});
}});
</script>
"""

# Append our control interface components directly into the map wrapper
m.get_root().html.add_child(folium.Element(clickthrough_script))
folium.LayerControl().add_to(m)

# m.save("clickthrough_tsp_route.html")
# print("Interactive clickthrough deployment map compiled successfully as 'clickthrough_tsp_route.html'.")
m

KeyError: '2'

In [37]:
import folium
import pandas as pd

coordinate_lookup = beach.set_index('Special Number')[['Latitude', 'Longitude']].to_dict('index')

port_lat = 37.7188
port_lon = 24.05663  

ordered_path_coordinates = []

for node in best_blueprint:
    lat = coordinate_lookup[node]['Latitude']
    lon = coordinate_lookup[node]['Longitude']
    ordered_path_coordinates.append([lat, lon])

# ==========================================
# 3. INITIALIZE THE WEB MAP
# ==========================================
# Center the map at the port
tsp_map = folium.Map(location=[port_lat, port_lon], zoom_start=9, control_scale=True)

# Add map styles (Street view and Satellite view toggles)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(tsp_map)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(tsp_map)

# ==========================================
# 4. DRAW THE SAILING LINES
# ==========================================
# Draw the continuous optimized route line connecting all nodes
folium.PolyLine(
    locations=ordered_path_coordinates,
    color="blue",
    weight=3,
    opacity=0.75,
    tooltip="Optimized Sailing Fleet Route"
).add_to(tsp_map)

# ==========================================
# 5. ADD MARKERS FOR STOPS
# ==========================================
# Add a special gold star marker for the Home Port
folium.Marker(
    location=[port_lat, port_lon],
    popup="<b>START/END HOME PORT</b>",
    icon=folium.Icon(color="red", icon="home", prefix="fa")
).add_to(tsp_map)

# Add standard circle markers for each beach stop along the sequence
for idx, node in enumerate(best_blueprint):
    if node == 'PORT':
        continue # Skip re-drawing the port as a standard circle
        
    lat = coordinate_lookup[node]['Latitude']
    lon = coordinate_lookup[node]['Longitude']
    
    # Label show the stop number sequence (e.g., Stop 1, Stop 2...)
    popup_text = f"<b>Stop #{idx}</b><br>Beach ID: {node}"
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        popup=folium.Popup(popup_text, max_width=200),
        color="black",
        weight=1,
        fill=True,
        fill_color="dodgerblue",
        fill_opacity=0.8
    ).add_to(tsp_map)

# Add the layer control toggle interface to the top right
folium.LayerControl().add_to(tsp_map)

# ==========================================
# 6. RENDER AND SAVE
# ==========================================
# tsp_map.save("optimized_tsp_route.html")
# print("Sailing route map successfully compiled! Open 'optimized_tsp_route.html' to inspect the path.")
tsp_map



## Interactive Map

In [9]:
# !pip install folium
import folium
from sklearn.cluster import DBSCAN
import matplotlib.cm as cm
import matplotlib.colors as colors


df = beach.copy()

coords = df[['Latitude', 'Longitude']].to_numpy()
coords_radians = np.radians(coords)

kms_per_radian = 6371.0 
max_distance_km = 30.0  # Max distance between beaches in the same cluster
epsilon = max_distance_km / kms_per_radian 

db = DBSCAN(eps=epsilon, min_samples=2, metric='haversine').fit(coords_radians)
df['Cluster_ID'] = db.labels_

# 3. Initialize the Interactive Map centered around your data's average location
center_lat = df['Latitude'].mean()
center_lon = df['Longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=9, control_scale=True)

# Add different map layers for auditing (Street view vs Satellite)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(m)

# 4. Generate a dynamic color palette for valid clusters
unique_clusters = set(df['Cluster_ID'])
n_clusters = len(unique_clusters) - (1 if -1 in unique_clusters else 0)

# Map dynamic hex colors to each unique cluster ID
colormap = cm.get_cmap('tab10', n_clusters + 1)
cluster_colors = {}
for idx, cluster in enumerate(sorted(list(unique_clusters))):
    if cluster == -1:
        cluster_colors[cluster] = '#ff0000' # Force Red for isolated noise points
    else:
        rgb = colormap(idx)[:3]
        cluster_colors[cluster] = colors.rgb2hex(rgb)

# 5. Plot the points onto the map layer
for _, row in df.iterrows():
    c_id = int(row['Cluster_ID'])
    special_num = row['Special Number']
    
    
    if c_id == -1:
        popup_text = f"<b>Beach:</b> {special_num}<br><span style='color:red;'><b>ISOLATED NOISE (ALL-INF)</b></span>"
        marker_color = cluster_colors[c_id]
        radius_size = 9
        fill_opacity_val = 0.8
    else:
        popup_text = f"<b>Beach:</b> {special_num}<br><b>Cluster ID:</b> {c_id}"
        marker_color = cluster_colors[c_id]
        radius_size = 7
        fill_opacity_val = 0.6

    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=radius_size,
        popup=folium.Popup(popup_text, max_width=250),
        color='black',
        weight=1,
        fill=True,
        fill_color=marker_color,
        fill_opacity=fill_opacity_val
    ).add_to(m)

folium.LayerControl().add_to(m)
m

/var/folders/jv/5h6fnlc51mg5n0fkcg1k6wvm0000gn/T/ipykernel_28618/2404343568.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap('tab10', n_clusters + 1)
